In [30]:
import requests
import pandas as pd
import os

from google.colab import drive
from dotenv import load_dotenv

### 1. 데이터 불러오기

1. env 데이터 불러오기
  - 키값, URL 데이터 값 불러오기
2. 키값과 URL 이용해 지역안전등급 데이터 불러오기
  - **데이터명**: [행정안전부_통계연보_지역안전등급](https://www.data.go.kr/data/15077976/openapi.do)
  - 설명
      - 기준연도 각 지역별로 교통사고, 화재, 범죄, 자연재해, 생활안전, 자살, 감염병 등의 등급 정보를 시도 단위로 제공\
      - 데이터 포맷: JSON + XML

In [31]:
# Google Drive 마운트
drive.mount('/content/drive')
env_file_path = '/content/drive/MyDrive/ColabNotebooks/kakao/env.text'

result = load_dotenv(env_file_path)
print("env 로드 여부:", result)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
env 로드 여부: True


In [32]:
api_url = os.getenv("REGIONAL_SAFETY_GRADE_URL")
api_key = os.getenv("REGIONAL_URL_SERVICE_KEY")

In [33]:
url = api_url

# 요청 변수 (Request Parameter)
#   서비스키, 페이지 번호, 한 페이지 결과 수, 호출문서
params = {
    "serviceKey": api_key,
    "pageNo": 1,
    "numOfRows": 100,
    "type": "json"
}

response = requests.get(url, params=params)
data = response.json()

print(data)


{'RegionalSafetyGrade': [{'head': [{'totalCount': 153}, {'numOfRows': '100', 'pageNo': '1', 'type': 'JSON'}, {'RESULT': {'resultCode': 'INFO-0', 'resultMsg': 'NOMAL SERVICE'}}]}, {'row': [{'bas_yy': '2016', 'regi': '서울', 'trffac': '1', 'fire': '2', 'crim': '5', 'natdsast': '1', 'comu_safe': '4', 'suicid': '3', 'infect': '3'}, {'bas_yy': '2016', 'regi': '부산', 'trffac': '2', 'fire': '3', 'crim': '4', 'natdsast': '5', 'comu_safe': '1', 'suicid': '5', 'infect': '5'}, {'bas_yy': '2016', 'regi': '대구', 'trffac': '3', 'fire': '3', 'crim': '3', 'natdsast': '2', 'comu_safe': '2', 'suicid': '4', 'infect': '4'}, {'bas_yy': '2016', 'regi': '인천', 'trffac': '2', 'fire': '4', 'crim': '2', 'natdsast': '4', 'comu_safe': '2', 'suicid': '4', 'infect': '3'}, {'bas_yy': '2016', 'regi': '광주', 'trffac': '4', 'fire': '1', 'crim': '4', 'natdsast': '2', 'comu_safe': '3', 'suicid': '2', 'infect': '4'}, {'bas_yy': '2016', 'regi': '대전', 'trffac': '3', 'fire': '2', 'crim': '3', 'natdsast': '4', 'comu_safe': '3', 'su

### 2. 데이터 변환

1. JSON 데이터에서 실제 지역안전등급 정보가 들어있는 row 부분만 추출
2. 추출한 row 데이터를 pandas DataFrame 형태로 변환
3. 문자열로 저장된 등급 데이터를 숫자(int) 형태로 변환

In [34]:
# JSON 데이터에서 실제 지역안전등급 정보가 들어있는 row 부분만 추출
rows = data["RegionalSafetyGrade"][1]["row"]

# DataFrame 생성
df = pd.DataFrame(rows)

# 등급 컬럼 목록
# 교통사고, 화재, 범죄, 자연재해, 생활안전, 자살, 감염병
grade_columns = ["trffac", "fire", "crim", "natdsast", "comu_safe", "suicid", "infect"]

# 문자열로 저장된 등급 데이터를 숫자(int) 형태로 변환
for col in grade_columns:
    df[col] = pd.to_numeric(df[col])

print(df)

   bas_yy regi  trffac  fire  crim  natdsast  comu_safe  suicid  infect
0    2016   서울       1     2     5         1          4       3       3
1    2016   부산       2     3     4         5          1       5       5
2    2016   대구       3     3     3         2          2       4       4
3    2016   인천       2     4     2         4          2       4       3
4    2016   광주       4     1     4         2          3       2       4
..    ...  ...     ...   ...   ...       ...        ...     ...     ...
95   2021   충북       3     2     4         0          3       2       4
96   2021   충남       4     3     2         0          4       4       3
97   2021   전북       3     3     2         0          2       3       2
98   2021   전남       5     4     3         0          4       3       4
99   2021   경북       4     3     1         0          3       3       5

[100 rows x 9 columns]


### 3. 데이터 그룹화

1. 지역(regi) 기준으로 데이터를 그룹화(groupby)
2. 이후 범죄 등급(crim) 컬럼만 선택
3. 각 지역별 범죄 등급의 평균(mean)을 계산

In [35]:
grouped_crim = df.groupby("regi")["crim"].mean()
print(grouped_crim)

regi
강원    3.666667
경기    3.333333
경남    3.000000
경북    1.666667
광주    3.166667
대구    2.333333
대전    3.833333
부산    4.333333
서울    4.666667
세종    1.000000
울산    2.000000
인천    2.666667
전남    1.833333
전북    1.666667
제주    5.000000
충남    2.833333
충북    3.833333
Name: crim, dtype: float64


### 4. 데이터 조건 필터링

- 지역(regi) 기준으로 데이터 그룹화
- 조건을 만족하는 그룹만 남김
    - 범죄 등급 평균이 4 이상
    - 교통사고 등급 평균이 2 이상
    - 화재 등급 평균이 2 이상

In [36]:
filtered = df.groupby("regi").filter(lambda x: (x['crim'].mean() >= 4) & (x['trffac'].mean() >= 2) & (x['fire'].mean() >= 2))

print(filtered)

   bas_yy regi  trffac  fire  crim  natdsast  comu_safe  suicid  infect
1    2016   부산       2     3     4         5          1       5       5
18   2017   부산       2     4     4         3          2       5       5
35   2018   부산       2     4     4         2          1       5       4
52   2019   부산       2     2     4         0          1       5       4
69   2020   부산       2     3     5         0          1       5       5
86   2021   부산       2     3     5         0          3       5       3
